# Direct Preference Optimization (DPO) Pipeline (DeepSeek-R1-7B)

## 🎯 What Are We Doing?
We are running Direct Preference Optimization (DPO) on top of the SFT-aligned `DeepSeek-R1-Distill-Qwen-7B` model using preference pairs (`chosen` vs `rejected`), followed by benchmark evaluation and artifact export.

## 💡 Why Are We Doing It?
While SFT teaches formatting and telegraphic syntax, DPO directly optimizes the model's preference margin to reward concise, accurate reasoning (`chosen`) over verbose or error-prone derivations (`rejected`), ensuring maximum brevity without sacrificing math precision.

## 🛠️ Code Source & Infrastructure
- **GitHub Repository:** [Hari31416/qwen-grug-finetune](https://github.com/Hari31416/qwen-grug-finetune.git)
- **Frameworks Used:** PyTorch, Hugging Face `transformers`, `peft` (LoRA), `trl` (`DPOTrainer`), `bitsandbytes` (4-bit NF4 quantization).
- **Target Hardware:** 2x NVIDIA T4 GPUs (Kaggle / Google Colab CUDA environment).

## 📊 Data Source
- **Hugging Face Dataset Repository:** [hari31416/qwen-grug-finetune](https://huggingface.co/datasets/hari31416/qwen-grug-finetune) (Preference dataset splits containing JSONL rows with `prompt`, `chosen`, and `rejected`).
- **Evaluation Benchmark:** [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k) (Grade School Math reasoning test split).

---

### Notebook Execution Workflow:
1. **DPO Hyperparameters**: Set preference optimization learning rate (`5e-7`), KL penalty (`beta=0.1`), and batch sizes.
2. **Environment & Dataset Setup**: Clone repository, load preference data (`data/dpo/train.jsonl`).
3. **Load SFT Reference Model**: Load 4-bit base model and apply baseline SFT LoRA weights.
4. **Execute DPO Training**: Run `run_dpo_training()` using Hugging Face TRL `DPOTrainer`.
5. **DPO GSM8K Evaluation**: Benchmark DPO model vs. Base and SFT models.
6. **Qualitative Preference Inspection**: Inspect how DPO further refines telegraphic brevity and accuracy.
7. **Export DPO Package**: Package DPO adapters into a downloadable ZIP archive.


## 1. Centralized DPO Hyperparameters & Configuration


In [ ]:
# ==========================================
# ⚙️ DPO EXPERIMENTAL HYPERPARAMETERS & CONFIG
# ==========================================

REPO_URL = "https://github.com/Hari31416/qwen-grug-finetune.git"
REPO_NAME = "qwen-grug-finetune"
MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
SFT_ADAPTER_PATH = "adapters/deepseek-r1-7b/20260804_040058/final_adapters"
DPO_DATA_DIR = "data/dpo"
DPO_OUTPUT_DIR = "adapters/deepseek-r1-7b/dpo"

# DPO Training Hyperparameters
DPO_EPOCHS = 1             # 1-2 epochs for preference optimization
DPO_BATCH_SIZE = 1         # Per-device batch size
DPO_GRAD_ACCUM = 8         # Gradient accumulation steps
DPO_LEARNING_RATE = 5e-7   # Learning rate (100x smaller than SFT)
DPO_BETA = 0.1             # KL divergence penalty weight
MAX_LENGTH = 1536          # Maximum sequence length
MAX_PROMPT_LENGTH = 512    # Maximum prompt length

EVAL_LIMIT = None         # Set to None for FULL benchmark evaluation (all 1,000 test samples), or set e.g. 50 for quick debugging
EVAL_BATCH_SIZE = 1        # Evaluation batch size


## 2. Environment Setup & Preference Data Verification


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%pip install -q peft trl bitsandbytes datasets accelerate huggingface_hub matplotlib seaborn pandas pyyaml


In [ ]:
import sys
if not os.path.exists("scripts") and not os.path.exists(f"{REPO_NAME}/scripts"):
    print(f"Cloning {REPO_URL} into workspace...")
    !git clone {REPO_URL}
    if os.path.exists(REPO_NAME):
        %cd {REPO_NAME}
elif os.path.exists(REPO_NAME) and os.path.exists(f"{REPO_NAME}/scripts"):
    %cd {REPO_NAME}

sys.path.append(".")
from scripts.cuda.cuda_utils import patch_transformers_lazy_imports
from scripts.cuda.download_data import download_hf_data
from scripts.create_dpo_dataset import generate_dpo_dataset

patch_transformers_lazy_imports()

# 1. Download base dataset from Hugging Face repository
print("Downloading dataset files from Hugging Face (hari31416/qwen-grug-finetune)...")
download_hf_data(output_dir="data")

# 2. Check and auto-generate DPO dataset from downloaded SFT data if missing
train_file = os.path.join(DPO_DATA_DIR, "train.jsonl")
if not os.path.exists(train_file):
    print(f"Generating DPO dataset at '{DPO_DATA_DIR}' from downloaded SFT format data...")
    generate_dpo_dataset(data_dir="data", dpo_dir=DPO_DATA_DIR)
else:
    print(f"✅ Found existing DPO dataset: {train_file}")


## 3. Execute DPO Fine-Tuning (`DPOTrainer`)


In [ ]:
from scripts.cuda.dpo_cuda import run_dpo_training

print("Starting DPO Fine-Tuning...")
dpo_trainer = run_dpo_training(
    model_arg=MODEL_ID,
    adapter_path=SFT_ADAPTER_PATH,
    dpo_data_dir=DPO_DATA_DIR,
    output_dir=DPO_OUTPUT_DIR,
    epochs=DPO_EPOCHS,
    batch_size=DPO_BATCH_SIZE,
    grad_accum=DPO_GRAD_ACCUM,
    learning_rate=DPO_LEARNING_RATE,
    beta=DPO_BETA,
    max_length=MAX_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
)


## 4. Benchmark Evaluation on DPO Model


In [ ]:
import torch
from scripts.cuda.cuda_utils import load_causal_lm_model, load_causal_lm_tokenizer
from scripts.cuda.eval_cuda import run_gsm8k_eval
from peft import PeftModel

tokenizer = load_causal_lm_tokenizer(MODEL_ID)
model = load_causal_lm_model(MODEL_ID, device_map="auto", torch_dtype=torch.float16)

dpo_adapter_path = os.path.join(DPO_OUTPUT_DIR, "final_dpo_adapters")
if os.path.exists(dpo_adapter_path):
    print("Evaluating DPO Model on GSM8K Benchmark...")
    dpo_model = PeftModel.from_pretrained(model, dpo_adapter_path)
    dpo_summary = run_gsm8k_eval(dpo_model, tokenizer, limit=EVAL_LIMIT, batch_size=EVAL_BATCH_SIZE, is_adapter=True)
else:
    print(f"DPO adapter path '{dpo_adapter_path}' not found.")


## 5. Export DPO Artifacts Package


In [ ]:
import zipfile

ZIP_FILE = "kaggle_dpo_artifacts.zip"
print(f"Creating downloadable DPO artifacts package: {ZIP_FILE}...")
with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zipf:
    if os.path.exists(DPO_OUTPUT_DIR):
        for root, dirs, files in os.walk(DPO_OUTPUT_DIR):
            for file in files:
                fp = os.path.join(root, file)
                zipf.write(fp, os.path.relpath(fp, "."))

if os.path.exists(ZIP_FILE):
    size_mb = os.path.getsize(ZIP_FILE) / (1024 * 1024)
    print(f"\n✅ Packaged DPO artifacts into '{ZIP_FILE}' ({size_mb:.2f} MB)")
